This file  used to check data quality, calculate metrics, and audit evaluator accuracy for Data Labelling stage.

Cell 1: Check for Empty Labels
Scans the project folders for missing data before any analysis begins.
Loops through the subdirectories, looks specifically for topic_labeled_updated.csv files, and prints a warning alongside sample IDs if a row has a pair_id but is missing its final_label.

Cell 2: Calculate Performance Metrics
Computes high-level accuracy statistics for each topic from the dataset.
Groups data by unique pair_id and calculates the labeler's accuracy (including breakdown by true/false classes and high-confidence scores) and the evaluator's accuracy using a confusion matrix.

Cell 3: Verify Evaluator Mistakes

Drills down into a single topic to catch errors made by the quality controller.
Filters for a specific topic (e.g., Gold Prices), compares the initial label directly against the final ground truth, and prints the top 10 rows where the evaluator's verdict was wrong.

In [ ]:
import os
import pandas as pd

root_folder = r"c:\Users\manoj\Thesis_project\outputs\data_labelling"

print(" INSPECTING DATA: CHECKING FOR EMPTY LABELS")
total_empty_rows = 0
files_checked_count = 0

for dirpath, dirnames, filenames in os.walk(root_folder):
    if dirpath == root_folder:
        continue
        
    for filename in filenames:
        # We only check the newly generated updated files
        if filename == 'topic_labeled_updated.csv':
            files_checked_count += 1
            file_path = os.path.join(dirpath, filename)
            folder_name = os.path.basename(dirpath)
            
            try:
                df = pd.read_csv(file_path)
                
                df.columns = df.columns.str.strip().str.lower()
                
                if 'final_label' in df.columns and 'pair_id' in df.columns:
                    # Find rows where pair_id exists but final_label is blank/NaN
                    # 'isna()' detects empty or missing values in Python
                    empty_mask = df['final_label'].isna() & df['pair_id'].notna()
                    df_empty = df[empty_mask]
                    
                    empty_count = len(df_empty)
                    
                    if empty_count > 0:
                        print(f" {folder_name}: Found {empty_count} rows missing a 'final_label'!")
                        # Print the first few missing pair_ids as examples
                        sample_ids = df_empty['pair_id'].head(5).tolist()
                        print(f"   └── Sample missing pair_ids: {sample_ids}")
                        total_empty_rows += empty_count
                    else:
                        print(f" {folder_name}: Perfect! No empty labels found.")
                else:
                    print(f" {folder_name}: Missing required columns for this check.")
                    
            except Exception as e:
                print(f" {folder_name}: Could not read file: {e}")

print("==================================================")
print("📊 FINAL AUDIT SUMMARY:")
print(f"   Total updated files checked: {files_checked_count}")
print(f"   Total empty 'final_label' rows found: {total_empty_rows}")
print("==================================================")


In [ ]:
import pandas as pd
import numpy as np
import os

topic_ids = [0, 1, 2, 3, 4, 5, 6, 7, 15, 17]

topic_names = {
    0: "Cybersecurity/Cloud",
    1: "Investor Conferences",
    2: "Analyst Notes/Stock Updates",
    3: "CEO/Officer Appointments",
    4: "Boeing/Airlines",
    5: "ECB/Eurozone",
    6: "Holding Company Notices",
    7: "Fed/Yellen/Rate",
    15: "Gold Prices",
    17: "European Shares",
}

results = []

for topic_id in topic_ids:

    path = f"topic_{topic_id}/topic_labeled_updated.csv"
    if not os.path.exists(path):
        continue

    df = pd.read_csv(path)

    # GROUP BY pair_id 
    grouped = df.groupby("pair_id").agg({
        "label": "first",
        "final_label": "first",
        "evaluator_verdict": "first",
        "confidence": "first"
    }).reset_index()

    total = len(grouped)
    total_cov = len(grouped["label"].dropna())

    # DOER METRICS
    mask = grouped["label"].notna() & grouped["final_label"].notna()
    comp = grouped[mask]

    doer_correct = (comp["label"] == comp["final_label"])
    doer_acc = doer_correct.mean() * 100 if len(comp) else 0

    true_mask = comp["final_label"] == True
    false_mask = comp["final_label"] == False

    true_acc = (
        (comp.loc[true_mask, "label"] == True).mean() * 100
        if true_mask.sum() else 0
    )

    false_acc = (
        (comp.loc[false_mask, "label"] == False).mean() * 100
        if false_mask.sum() else 0
    )

    # CONFIDENCE 4–5 ONLY
    conf_mask = mask & grouped["confidence"].isin([4, 5])
    conf = grouped[conf_mask]

    conf_acc = (
        (conf["label"] == conf["final_label"]).mean() * 100
        if len(conf) else 0
    )

    # EVALUATOR 

    eval_mask = (
        mask &
        grouped["evaluator_verdict"].isin(["correct", "incorrect"])
    )

    eval_df = grouped[eval_mask]

    if len(eval_df) > 0:

        doer_correct_eval = eval_df["label"] == eval_df["final_label"]

        evaluator_says_correct = eval_df["evaluator_verdict"] == "correct"
        evaluator_says_incorrect = eval_df["evaluator_verdict"] == "incorrect"

        TP = (doer_correct_eval & evaluator_says_correct).sum()
        TN = (~doer_correct_eval & evaluator_says_incorrect).sum()
        FP = (~doer_correct_eval & evaluator_says_correct).sum()
        FN = (doer_correct_eval & evaluator_says_incorrect).sum()

        eval_acc = (TP + TN) / len(eval_df) * 100

        eval_coverage = len(eval_df) / len(grouped) * 100

    else:
        TP = TN = FP = FN = 0
        eval_acc = 0
        eval_coverage = 0

    # STORE RESULTS
    results.append({
        "Topic": topic_id,
        "Name": topic_names.get(topic_id, "Unknown"),
        "Docs": total,
        "Cov": total_cov,

        "Doer%": round(doer_acc, 2),
        "True%": round(true_acc, 2),
        "False%": round(false_acc, 2),

        "Conf(4-5)%": round(conf_acc, 2),

        "Eval Acc%": round(eval_acc, 2),
        "Eval Cov%": round(eval_coverage, 2),

        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN
    })

# FINAL TABLE
summary_df = pd.DataFrame(results)

print("\n" + "=" * 120)
print("FINAL PAIR_ID-LEVEL EVALUATION TABLE (CORRECTED)")
print("=" * 120)

print(summary_df.to_string(index=False))

In [ ]:
topic_id = 15

df = pd.read_csv(f"topic_{topic_id}/topic_labeled_updated.csv")

df = df.dropna(subset=["pair_id"])

grouped = df.groupby("pair_id").agg({
    "label": "first",
    "final_label": "first",
    "evaluator_verdict": "first",
    "confidence": "first"
}).reset_index()

valid = grouped["label"].notna() & grouped["final_label"].notna()
dfv = grouped[valid]
# 1. Only keep rows where the evaluator actually gave a real grade
valid_verdicts = ['correct', 'incorrect']
dfv = dfv[dfv['evaluator_verdict'].isin(valid_verdicts)]

# 2. Run your original check on these true grades
doer_correct = dfv['label'] == dfv['final_label']
wrong_eval = dfv[ ( (doer_correct & (dfv['evaluator_verdict'] != 'correct')) | 
                    (~doer_correct & (dfv['evaluator_verdict'] != 'incorrect')) ) ]


print("Wrong evaluator predictions:", len(wrong_eval))

print(
    wrong_eval[[
        "pair_id",
        "label",
        "final_label",
        "evaluator_verdict",
        "confidence"
    ]].iloc[0:10]
)